# M1.1: Energy time series — getting and validating ENTSO-E data

## Objectives

At the end of this notebook you should be able to:

- explain what the ENTSO-E Transparency Platform publishes and how to access it with a security token
- download Croatian load data with `entsoe-py`, keeping the token out of Git
- handle time zones and daylight-saving time correctly (23- and 25-hour days)
- check an energy time series for duplicates, gaps and implausible values — and prove the checks work
- resample power data correctly: **average** MW, never sum it, and compute energy as MW × hours

Your data engineering is already strong, so this notebook focuses on what is special about energy data. Every model in this course is only as good as its input: a missing week of load data or a doubled hour after a clock change will flow straight into prices and investment decisions without any solver complaining.

In [1]:
import os

import numpy as np
import pandas as pd

### What the Transparency Platform publishes

The [ENTSO-E Transparency Platform](https://transparency.entsoe.eu/) is the central EU source for electricity data, published by the transmission system operators. The datasets you will use most:

| Dataset | Unit | Used in |
|---|---|---|
| Actual total load | MW | demand (this lesson, P1) |
| Actual generation per production type | MW | supply mix, renewables (P1, P2) |
| Day-ahead prices | €/MWh | markets (Module 02) |
| Cross-border physical flows | MW | interconnection (Module 04) |

Data are published per **bidding zone**. Croatia is a single zone with the code `10YHR-HEP------M`; `entsoe-py` lets you simply write `"HR"`. Values are average **power** over each interval (MW), not energy — keep that in mind for resampling below.

### Getting and storing a security token

The API needs a free security token:

1. Register on the [Transparency Platform](https://transparency.entsoe.eu/) and verify your email.
2. Email `transparency@entsoe.eu` with **RESTful API access** as the subject and your registered email address in the body.
3. Access is usually granted within 3 working days. Then log in, open **My Account** and generate a token.

Store it in a `.env` file next to your notebooks (`ENTSOE_API_KEY=...`) and make sure `.env` is in `.gitignore` — a token pushed to GitHub is a leaked credential. The next cell loads it if present, and the notebook still runs without it.

In [2]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # pip install python-dotenv to read .env files

ENTSOE_API_KEY = os.getenv("ENTSOE_API_KEY")
HAS_TOKEN = bool(ENTSOE_API_KEY)
print("ENTSO-E token found" if HAS_TOKEN else "No token yet - real-data cells will be skipped")

No token yet - real-data cells will be skipped


### Downloading real Croatian load

`entsoe-py` wraps the XML API and returns pandas objects. Start and end must be **time-zone-aware** timestamps. This cell downloads one week of Croatian actual load when a token is available.

In [3]:
if HAS_TOKEN:
    from entsoe import EntsoePandasClient

    client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
    start = pd.Timestamp("2024-01-08", tz="Europe/Zagreb")
    end = pd.Timestamp("2024-01-15", tz="Europe/Zagreb")
    hr_load = client.query_load("HR", start=start, end=end)
    print(hr_load.head())
    print(f"{len(hr_load)} rows, columns: {list(hr_load.columns)}, index tz: {hr_load.index.tz}")
else:
    print("Skipped: add ENTSOE_API_KEY to .env to download real data.")

Skipped: add ENTSOE_API_KEY to .env to download real data.


Look at the index: timestamps carry the Croatian time zone. Also check the spacing between rows — some zones publish hourly, others every 15 minutes, and a zone can switch resolution over time. Never assume; always check (`hr_load.index.to_series().diff().value_counts()`).

### Time zones and daylight-saving time

The API works in UTC; markets and people think in local time. Daylight-saving time makes local days uneven: in Croatia the last Sunday of March has **23 hours**, the last Sunday of October **25**. Code that assumes 24 hours per day silently shifts or drops data twice a year.

In [4]:
for day in ["2025-03-30", "2025-06-15", "2025-10-26"]:
    hours = pd.date_range(day, periods=2, freq="D", tz="Europe/Zagreb")
    n = int((hours[1] - hours[0]) / pd.Timedelta("1h"))
    print(f"{day}: {n} hours in Europe/Zagreb")

2025-03-30: 23 hours in Europe/Zagreb
2025-06-15: 24 hours in Europe/Zagreb
2025-10-26: 25 hours in Europe/Zagreb


**Rule for this course:** store and compute in **UTC**, convert to `Europe/Zagreb` only for display or for features like "hour of day". UTC has no gaps or repeated hours, so every interval is exactly one hour long.

### A teaching sample with known defects

To learn validation you need data whose problems you *know*. The function below builds a **synthetic** two-week series shaped like ENTSO-E load — it is not real Croatian data — and plants four defects that are common in real downloads: a gap, a duplicated timestamp with a revised value, a zero reading and a spike.

In [5]:
def make_teaching_sample(seed: int = 42) -> pd.Series:
    """SYNTHETIC hourly load (MW) shaped like ENTSO-E 'Actual Load' data, with planted defects.

    Not real Croatian data: a daily and weekly pattern around ~2,000 MW plus noise,
    used only to prove that each validation check catches the defect it targets.
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-03-24", periods=14 * 24, freq="h", tz="UTC")
    hour = idx.tz_convert("Europe/Zagreb").hour
    weekend = idx.tz_convert("Europe/Zagreb").dayofweek >= 5
    load = (2000 + 350 * np.sin((hour - 7) / 24 * 2 * np.pi)
            - 180 * weekend + rng.normal(0, 40, len(idx)))
    s = pd.Series(load.round(1), index=idx, name="load_MW")

    s = s.drop(s.index[100:105])                     # defect 1: 5-hour gap
    s = pd.concat([s, s.iloc[[150]] + 75]).sort_index(kind="stable")  # defect 2: duplicate, revised value last
    s.iloc[200] = 0.0                                # defect 3: zero reading
    s.iloc[250] = s.iloc[250] * 5                    # defect 4: spike
    return s

sample = make_teaching_sample()
sample.describe().round(1)

count      332.0
mean      1968.9
std        582.4
min          0.0
25%       1716.2
50%       1945.0
75%       2189.1
max      11188.5
Name: load_MW, dtype: float64

Already `describe()` hints at trouble: a minimum of 0 MW and a maximum far above the rest. Summary statistics are a quick first look, but they cannot tell you *where* the problems are, and they miss gaps and duplicates entirely. You need explicit checks.

### Check 1: duplicate timestamps

In [6]:
print(f"{sample.index.duplicated().sum()} duplicated timestamp(s)")
sample[sample.index.duplicated(keep=False)]

1 duplicated timestamp(s)


2025-03-30 11:00:00+00:00    2240.7
2025-03-30 11:00:00+00:00    2315.7
Name: load_MW, dtype: float64

One hour appears twice with different values; the second row is the later publication. In ENTSO-E data this usually means a **revised** value was published. A common, defensible rule is to keep the last value — but write the rule down, because it is a modelling decision.

### Check 2: gaps against a complete index

Build the index the data *should* have, from first to last timestamp at the expected frequency, and compare.

In [7]:
expected = pd.date_range(sample.index.min(), sample.index.max(), freq="h")
missing = expected.difference(sample.index)
print(f"expected {len(expected)} hours, missing {len(missing)}")
missing

expected 336 hours, missing 5


DatetimeIndex(['2025-03-28 04:00:00+00:00', '2025-03-28 05:00:00+00:00',
               '2025-03-28 06:00:00+00:00', '2025-03-28 07:00:00+00:00',
               '2025-03-28 08:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

Five consecutive hours are missing. Short gaps (a few hours) can usually be interpolated; long gaps cannot — you would be inventing data. Where to draw that line is another decision to document.

### Check 3: physically plausible values

Croatian load sits roughly between 1,000 and 3,500 MW (average ~2,000 MW, from the EE-1.1 exercise). A range check uses that knowledge. It is deliberately loose: its job is to catch readings that are physically impossible, not merely unusual.

In [8]:
lower_MW, upper_MW = 500, 4500
out_of_range = sample[(sample < lower_MW) | (sample > upper_MW)]
out_of_range

2025-04-01 12:00:00+00:00        0.0
2025-04-03 14:00:00+00:00    11188.5
Name: load_MW, dtype: float64

Both the zero reading and the spike are caught. A zero is a classic sign of a failed meter or a missing value written as 0 — a country's load never drops to zero. Treat it as missing, not as a measurement.

### Resampling power correctly

ENTSO-E values are **average power** (MW) over each interval. To go from 15-minute to hourly data you therefore **average** — four 15-minute readings of 2,000 MW are one hour at 2,000 MW, not 8,000 MW. Energy is power × time: each 15-minute value contributes MW × 0.25 h.

In [9]:
q = pd.Series([1900.0, 2000.0, 2100.0, 2000.0] * 2,
              index=pd.date_range("2025-01-01", periods=8, freq="15min", tz="UTC"), name="load_MW")

hourly_MW = q.resample("h").mean()        # correct
wrong_MW = q.resample("h").sum()          # a common bug
energy_MWh = (q * 0.25).sum()             # MW x h

print(hourly_MW.to_string(), "\n")
print(f"wrong 'hourly' values: {wrong_MW.tolist()} MW  <- 4x too high")
print(f"energy over 2 hours: {energy_MWh:,.0f} MWh = {hourly_MW.sum():,.0f} MWh from hourly averages")

2025-01-01 00:00:00+00:00    2000.0
2025-01-01 01:00:00+00:00    2000.0
Freq: h 

wrong 'hourly' values: [8000.0, 8000.0] MW  <- 4x too high
energy over 2 hours: 4,000 MWh = 4,000 MWh from hourly averages


Averaging keeps the power level right, and the energy is the same whether you compute it from 15-minute or hourly data (4,000 MWh over two hours) — a good built-in check. The summed version reports 8,000 MW for a system that never exceeded 2,100 MW: wrong by a factor of four, and nothing would warn you. This is the MW vs. MWh lesson from EE-1.1 in its most common real-world form.

## Check your understanding

**Time and units**

1. How many hours does 2025 have in `Europe/Zagreb` local time? In UTC? Check with `pd.date_range`.
2. An hourly series in UTC averages 2,055 MW over a year. How much energy is that in TWh? Compare with the EE-1.1 exercise.

**Validation**

1. In the duplicate check, why does keeping the *last* value make sense for ENTSO-E data? When might it be wrong?
2. Change the range check to `lower_MW = 1500`. Which values are flagged now, and are they really errors? What does that teach you about choosing thresholds?

**Thinking ahead**

The exercise turns these checks into reusable functions, adds two more (spikes and stuck meters), and applies them to a full year of real Croatian load. Before starting: which defect do you expect to be most common in real ENTSO-E data, and why?